# The Computer Vision Project — CIFAR-10 Classification

This notebook works through the case introduced in *The Computer Vision Project*:
- **Task:** Classification problem with 10 classes (CIFAR-10 dataset)
- **Images:** Colorful (RGB), 32x32 in size
- **Data:** 50k images available for training, 10k for testing
- **Classes are mutually exclusive** (e.g. no overlap between `automobile` and `truck`)

We'll go through the same essential steps as the `CNN_classification` walkthrough: load & explore the data, preprocess it, build a model on top of a pretrained network, train it, and evaluate its performance.

## 1. Setup

In [ ]:
!pip install keras tensorflow

In [ ]:
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Load the Data

CIFAR-10 is available directly through `tensorflow.keras.datasets`.

In [ ]:
# Data load
(train_imgs, train_labels), (test_imgs, test_labels) = cifar10.load_data()

## 3. EDA (Exploratory Data Analysis)

In [ ]:
# Class names, in label order (0-9)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# Data Checkout
print("The number of training examples is: ", train_imgs.shape[0])
print("The number of test examples is: ", test_imgs.shape[0])
print("The size of every img is: ", train_imgs.shape[1:])
num_classes = len(np.unique(train_labels))
print("The number of classes is: ", num_classes)

In [ ]:
# cifar10 labels come back as shape (n, 1) - flatten them to (n,)
# So flattening aligns the labels with the shape the model/metrics expect.
train_labels = train_labels.flatten()
test_labels = test_labels.flatten()

In [ ]:
# printing out some samples
figure, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(train_imgs[7, :, :])
ax1.set_title("Ground truth is: {}".format(class_names[train_labels[7]]))

ax2.imshow(train_imgs[20, :, :])
ax2.set_title("Ground truth is: {}".format(class_names[train_labels[20]]))

In [ ]:
# One row of sample images per class, like in the project slides
fig, axes = plt.subplots(num_classes, 8, figsize=(12, 14))

for class_idx in range(num_classes):
    class_img_indices = np.where(train_labels == class_idx)[0][:8]
    for col, img_idx in enumerate(class_img_indices):
        ax = axes[class_idx, col]
        ax.imshow(train_imgs[img_idx])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(class_names[class_idx])
    axes[class_idx, 0].set_title(class_names[class_idx], loc='left', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Limiting the Dataset

Using all 50k training images gives the best performance but takes more time and compute. **To save time, let's limit the training data to 10k only.** (Note: the test set already only has 10k images, so it doesn't need limiting.)

In [ ]:
# Getting only 10k from the training portion of data
n = 10000

train_imgs = train_imgs[:n]
train_labels = train_labels[:n]

## 5. Data Preprocessing

### 5.1 Normalize pixel values
Neural nets expect normalized images. We move pixel values from the `0..255` range to `0..1`.

In [ ]:
# Convert images to float and scale from 0..255 to 0..1
train_data = train_imgs.astype('float32')
test_data = test_imgs.astype('float32')

train_data /= 255
test_data /= 255

### 5.2 One-hot encode the labels

In [ ]:
# Converting labels to one-hot encoding form
train_labels_one_hot = to_categorical(train_labels, num_classes)
test_labels_one_hot = to_categorical(test_labels, num_classes)

In [ ]:
print("Before encoding", train_labels[5])
print("After encoding", train_labels_one_hot[5])

## 6. Model Selection

### 6.1 Import a pretrained network

We use `ResNet50` pretrained on ImageNet as our feature extractor:
- `weights='imagenet'` — import the network pretrained on ImageNet
- `include_top=False` — drop off the top (classification) part of the network
- `input_shape` — matches our CIFAR-10 image size, `(32, 32, 3)`

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)
)

# Freeze the pretrained base so we only train the new top layers
base_model.trainable = False

### 6.2 Build the top of the network

We add:
- A `GlobalAveragePooling2D` layer to turn the feature maps into a single vector
- Two hidden `Dense` layers (128, then 64 neurons — powers of two, first layer larger)

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dense(64, activation='relu')(x)

### 6.3 Add the output layer

The output layer has one neuron per class, with a `softmax` activation for multi-class classification.

In [ ]:
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.summary()

## 7. Compile, Train, Evaluate

### 7.1 Compile the model

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### 7.2 Train the model

In [ ]:
history = model.fit(
    train_data, train_labels_one_hot,
    batch_size=64,
    epochs=20,
    verbose=1,
    validation_data=(test_data, test_labels_one_hot)
)

### 7.3 Evaluate on the test data

In [ ]:
[test_loss, test_acc] = model.evaluate(test_data, test_labels_one_hot)
print("Evaluation result on Test Data : Loss = {}, accuracy = {}".format(test_loss, test_acc))

## 8. Visualizing Training Performance

In [ ]:
plt.figure(figsize=[12, 6])
plt.plot(history.history['loss'], 'r', linewidth=3.0)
plt.plot(history.history['val_loss'], 'b', linewidth=3.0)
plt.legend(['Training loss', 'Validation loss'], fontsize=14)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.title('Loss curves', fontsize=14)
plt.grid()

- Both train & val loss decrease and flatten out → good fit. The model is learning and generalizing.
- Train loss low, val loss high & increasing → overfitting. Model memorizes training data but fails on unseen data.
- Both high, barely decreasing → underfitting. Model is too simple or learning rate is too low.
- Val loss drops then rises, train keeps falling → classic overfitting; stop early ("early stopping"), add dropout, more data, or regularization.

Key idea: loss = error, so lower is better. The gap between train and val loss tells you how much the model generalizes vs. memorizes.

In [ ]:
plt.figure(figsize=[12, 6])
plt.plot(history.history['accuracy'], 'r', linewidth=3.0)
plt.plot(history.history['val_accuracy'], 'b', linewidth=3.0)
plt.legend(['Training Accuracy', 'Validation Accuracy'], fontsize=14)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.title('Accuracy curves', fontsize=14)
plt.grid()

- Both train & val accuracy rise and plateau high (e.g. 0.90+) → good fit. Model is learning and generalizing.
- Train accuracy high, val accuracy low or falling → overfitting. Model memorizes training data but fails on unseen data.
- Both low and barely rising → underfitting. Model too simple or learning rate too low.

Key idea: accuracy = % correct, so higher is better. The gap between train and val accuracy tells you how much the model generalizes vs. memorizes — a big gap means overfitting.

Accuracy vs. loss: loss is smoother and more sensitive to small changes (useful across all epochs); accuracy is an intuitive 0–1 score but can be noisy with small batches and useless if classes are imbalanced.

## 9. Making Predictions

In [ ]:
# Predict on the full test set
test_predictions = model.predict(test_data)
predicted_labels = np.argmax(test_predictions, axis=1)

In [ ]:
# Check a single prediction against ground truth
idx = 8
single_pred = model.predict(test_data[[idx], :])
predicted_class = np.argmax(single_pred)

plt.imshow(test_imgs[idx])
plt.title(f"Predicted: {class_names[predicted_class]} | Ground truth: {class_names[test_labels[idx]]}")
plt.axis('off')
plt.show()

## 11. Alternative Pretrained Base: `EfficientNet`

Next step from the list above: let's swap `ResNet50` for `EfficientNet` and follow the exact same steps, to see if we get better accuracy.

We use `EfficientNetB0` (the smallest, fastest variant) pretrained on ImageNet as our feature extractor:
- `weights='imagenet'` — import the network pretrained on ImageNet
- `include_top=False` — drop off the top (classification) part of the network
- `input_shape` — matches our CIFAR-10 image size, `(32, 32, 3)`

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

### 11.1 Build the model

Same architecture as before — a frozen pretrained base plus a small head:
- A `GlobalAveragePooling2D` layer to turn the feature maps into a single vector
- Two hidden `Dense` layers (128, then 64 neurons)
- A `softmax` output layer with one neuron per class

In [ ]:
# Load EfficientNetB0 pretrained on ImageNet; include_top=False drops
# the original classification head, keeping only feature-extraction layers
efficient_base = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(32, 32, 3)  # 32x32 RGB images (smaller than EfficientNet's 224x224)
)

# Freeze the pretrained weights: during training only the new top layers update
efficient_base.trainable = False

x = efficient_base.output          # feature maps from the frozen base
x = GlobalAveragePooling2D()(x)    # collapse each feature map to 1 scalar per channel
x = Dense(128, activation='relu')(x)  # hidden layer learning feature combinations
x = Dense(64, activation='relu')(x)   # another hidden layer
predictions = Dense(num_classes, activation='softmax')(x)  # class probabilities

# Wrap base + new head into one model
efficient_model = Model(inputs=efficient_base.input, outputs=predictions)
efficient_model.summary()  # print architecture and param counts

### 11.2 Compile the model

In [ ]:
efficient_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

### 11.3 Train the model

In [ ]:
efficient_history = efficient_model.fit(
    train_data, train_labels_one_hot,
    batch_size=64,
    epochs=20,
    verbose=1,
    validation_data=(test_data, test_labels_one_hot)
)

### 11.4 Evaluate on the test data

In [ ]:
[efficient_test_loss, efficient_test_acc] = efficient_model.evaluate(test_data, test_labels_one_hot)
print("EfficientNet evaluation result on Test Data : Loss = {}, accuracy = {}".format(efficient_test_loss, efficient_test_acc))

### 11.5 Visualizing Training Performance

In [ ]:
plt.figure(figsize=[12, 6])
plt.plot(efficient_history.history['loss'], 'r', linewidth=3.0)
plt.plot(efficient_history.history['val_loss'], 'b', linewidth=3.0)
plt.legend(['Training loss', 'Validation loss'], fontsize=14)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.title('EfficientNet loss curves', fontsize=14)
plt.grid()

In [ ]:
plt.figure(figsize=[12, 6])
plt.plot(efficient_history.history['accuracy'], 'r', linewidth=3.0)
plt.plot(efficient_history.history['val_accuracy'], 'b', linewidth=3.0)
plt.legend(['Training Accuracy', 'Validation Accuracy'], fontsize=14)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Accuracy', fontsize=14)
plt.title('EfficientNet accuracy curves', fontsize=14)
plt.grid()

### 11.6 Making Predictions

In [ ]:
# Predict on the full test set
efficient_test_predictions = efficient_model.predict(test_data)
efficient_predicted_labels = np.argmax(efficient_test_predictions, axis=1)

In [ ]:
# Check a single prediction against ground truth
idx = 8
efficient_single_pred = efficient_model.predict(test_data[[idx], :])
efficient_predicted_class = np.argmax(efficient_single_pred)

plt.imshow(test_imgs[idx])
plt.title(f"Predicted: {class_names[efficient_predicted_class]} | Ground truth: {class_names[test_labels[idx]]}")
plt.axis('off')
plt.show()

### 11.7 Comparison: `ResNet50` vs `EfficientNet`

In [ ]:
print("ResNet50 test accuracy   : {:.4f}".format(test_acc))
print("EfficientNet test accuracy: {:.4f}".format(efficient_test_acc))

if efficient_test_acc > test_acc:
    print("EfficientNet does better than ResNet50 by {:.4f}.".format(efficient_test_acc - test_acc))
elif efficient_test_acc < test_acc:
    print("ResNet50 does better than EfficientNet by {:.4f}.".format(test_acc - efficient_test_acc))
else:
    print("Both pretrained bases give the same test accuracy.")

### 12. Sequential-API variant

Instead of the Functional API (`inputs=...`, `outputs=...`), we can build the same model with `tf.keras.Sequential` by passing the frozen `EfficientNetB0` base in as the first layer:
- `layers[0]` is the pretrained base, so we freeze it by setting `trainable = False`
- Same head as before (GlobalAveragePooling2D + Dense 128 + Dense 64 + softmax)

In [ ]:
from tensorflow.keras.models import Sequential

efficient_model_seq = Sequential([
    EfficientNetB0(weights='imagenet', include_top=False,
                   input_shape=(32, 32, 3)),
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# Freeze the pretrained base (first layer) so we only train the new top layers
efficient_model_seq.layers[0].trainable = False

efficient_model_seq.summary()

### 12.1 Compile, Train, Evaluate the Sequential model

In [ ]:
efficient_model_seq.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
efficient_seq_history = efficient_model_seq.fit(
    train_data, train_labels_one_hot,
    batch_size=64,
    epochs=10,
    verbose=1,
    validation_data=(test_data, test_labels_one_hot)
)

### 12.2 Final Comparison: `ResNet50` vs `EfficientNet` vs `EfficientNet (Sequential)`

In [ ]:
[efficient_seq_test_loss, efficient_seq_test_acc] = efficient_model_seq.evaluate(test_data, test_labels_one_hot)
print("EfficientNet (Sequential) evaluation result on Test Data : Loss = {}, accuracy = {}".format(efficient_seq_test_loss, efficient_seq_test_acc))

In [ ]:
results = {
    'ResNet50': test_acc,
    'EfficientNet (Functional)': efficient_test_acc,
    'EfficientNet (Sequential)': efficient_seq_test_acc,
}

for name, acc in results.items():
    print("{:28s} test accuracy: {:.4f}".format(name, acc))

best_name = max(results, key=results.get)
print("\nBest pretrained base: {} with {:.4f} accuracy.".format(best_name, results[best_name]))